In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os

In [ ]:
nc_files = sorted(glob.glob("C:/Users/aneli/Documents/LOC/PC_casa/Dataset/**/*.nc", recursive=True))

In [ ]:
def subset_region(ds):
    # Ajustar longitudes se estiverem em 0–360
    if ds.lon.max() > 180:
        ds = ds.assign_coords(
            lon=((ds.lon + 180) % 360) - 180
        )

    # Criar máscara apenas para a dimensão n (perfis)
    mask = (
        (ds.lat <= 10) & (ds.lat >= -60) &
        (ds.lon >= -60) & (ds.lon <= -10)
    )

    if "Nobs" in ds.dims:
        ds = ds.isel(Nobs=mask)

    return ds
    
def check_staircase_exists(ml_h, gl_h): #verifica se os valores são fisicamente válidos (n só se existem)

    valid_ml = np.sum(~np.isnan(ml_h) & (ml_h > 0))
    valid_gl = np.sum(~np.isnan(gl_h) & (gl_h > 0))

    return (valid_ml >= 1) and (valid_gl >= 1)
    
def detect_staircases(ds):

    profile_dim = ds.lat.dims[0]
    n_profiles = ds.sizes[profile_dim] #p n ter problema d confundir "n" e "Nobs"

    staircase_sf = np.zeros(n_profiles, dtype=bool)
    staircase_dc = np.zeros(n_profiles, dtype=bool)

    for i in range(n_profiles):

        # SALT FINGER
        ml_mask_sf = ds.mask_ml_sf_layer.isel({profile_dim: i}) > 0
        gl_mask_sf = ds.mask_gl_sf_layer.isel({profile_dim: i}) > 0

        ml_h_sf = ds.ml_h.isel({profile_dim: i}).values[ml_mask_sf.values] #pega os valores onde ml_mask_sf é True 
        gl_h_sf = ds.gl_h.isel({profile_dim: i}).values[gl_mask_sf.values]

        if len(ml_h_sf) > 0 and len(gl_h_sf) > 0:
            staircase_sf[i] = check_staircase_exists(
                ml_h_sf, gl_h_sf
            )

        # DIFFUSIVE CONVECTION
        ml_mask_dc = ds.mask_ml_dc_layer.isel({profile_dim: i}) > 0
        gl_mask_dc = ds.mask_gl_dc_layer.isel({profile_dim: i}) > 0

        ml_h_dc = ds.ml_h.isel({profile_dim: i}).values[ml_mask_dc.values]
        gl_h_dc = ds.gl_h.isel({profile_dim: i}).values[gl_mask_dc.values]

        if len(ml_h_dc) > 0 and len(gl_h_dc) > 0:
            staircase_dc[i] = check_staircase_exists(
                ml_h_dc, gl_h_dc
            )

    ds["staircase_sf"] = (profile_dim, staircase_sf)
    ds["staircase_dc"] = (profile_dim, staircase_dc)

    return ds

In [ ]:
shape_path = r"C:/seu_caminho/shape.shp"

gdf_sismic = gpd.read_file(shape_path)

# garantir lat/lon
gdf_sismic = gdf_sismic.to_crs(epsg=4326)

In [ ]:
for file in nc_files:

    ds = xr.open_dataset(file)
    ds = subset_region(ds)
    n_region = ds.sizes.get("Nobs", 0)

    if n_region == 0:
        continue
        
    ds = detect_staircases(ds)

    sf_mask = ds.staircase_sf.values
    dc_mask = ds.staircase_dc.values

    #SEPARAÇÃO DOS REGIMES

    sf_pure_mask = sf_mask & (~dc_mask)
    dc_pure_mask = dc_mask & (~sf_mask)
    mixed_mask = sf_mask & dc_mask
    stair_mask = sf_mask | dc_mask

    # PONTOS ESPACIAIS
    lat = ds.lat.values
    lon = ds.lon.values
    lat_round = np.round(lat, 2)
    lon_round = np.round(lon, 2)

    # SF PURO
    for la, lo in zip(lat_round[sf_pure_mask], lon_round[sf_pure_mask]):
        all_points_sf.append((lo, la))

    # DC PURO
    for la, lo in zip(lat_round[dc_pure_mask], lon_round[dc_pure_mask]):
        all_points_dc.append((lo, la))

    # MISTO
    for la, lo in zip(lat_round[mixed_mask], lon_round[mixed_mask]):
        all_points_mixed.append((lo, la))
        
unique_points_sf = set(all_points_sf)
unique_points_dc = set(all_points_dc)
unique_points_mixed = set(all_points_mixed)


In [ ]:
sf_lons, sf_lats = zip(*unique_points_sf) if unique_points_sf else ([], [])
dc_lons, dc_lats = zip(*unique_points_dc) if unique_points_dc else ([], [])
mixed_lons, mixed_lats = zip(*unique_points_mixed) if unique_points_mixed else ([], [])

In [ ]:
fig = plt.figure(figsize=(10,8))

ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-60, -10, -60, 10], crs=ccrs.PlateCarree())

# mapa base
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.COASTLINE)

gl = ax.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False

# extensão (ajuste conforme necessário)
ax.set_extent([-70, -20, -50, 0])

# LINHAS SÍSMICAS (SHAPEFILE)
ax.add_geometries(
    gdf_sismic.geometry,
    crs=ccrs.PlateCarree(),
    edgecolor="lightgray",
    facecolor="none",
    linewidth=1
)

# PONTOS ARGO
ax.scatter(
    lon_all,
    lat_all,
    s=4,
    color="gray",
    transform=ccrs.PlateCarree(),
    label="Perfis"
)

# SF
ax.scatter(sf_lons, sf_lats,
    s=12,
    color="blue",
    transform=ccrs.PlateCarree(),
    label="SF"
)

# DC
ax.scatter(
    dc_lons, dc_lats,
    s=12,
    color="green",
    transform=ccrs.PlateCarree(),
    label="DC"
)

# Mixed
ax.scatter(
    mixed_lons, mixed_lats,
    s=15,
    color="purple",
    transform=ccrs.PlateCarree(),
    label="Mixed"
)

# Legenda externa
ax.legend(
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False,
    markerscale=3
)

plt.title("Distribuição de Perfis Argo e Linhas Sísmicas")
#plt.legend()
plt.show()